# WeightedKgBlend — ProbCBR (Updated Splits)
Probabilistic Case-Based Reasoning (Das et al., EMNLP Findings 2020).

Runs on **splits_updated** — 5-fold CV built from `mind_updated.tsv` (+1,391 new DrugCentral indication edges, 6,761 total).

Pipeline:
1. Entity representation — binary outgoing relation-type vectors
2. Entity clustering — KMeans on cosine-normalized vectors
3. Path mining — BFS from each training drug via biologically meaningful relations
4. Path statistics — per-cluster path prior P(path|cluster) and precision P(answer|path,cluster)
5. Inference — cosine k-NN within cluster → aggregate prior×precision → execute paths from query drug
6. Output — ranked candidates with mechanistic path strings

Reads splits from `wkb-splits-updated` dataset.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, time
from collections import defaultdict

INDICATION_REL = 'indication'
SPLITS_TO_RUN  = list(range(5))  # all 5 slices
DEBUG_N        = None             # full run (set to 50 for quick debug)

# Biologically meaningful relations for BFS path traversal (MIND-specific).
# Only mechanistic intermediates — no direct drug→disease treatment relations
BIO_RELATIONS = {
    # Drug → Gene
    'inhibits_CinG', 'activates_CaG', 'affects_CafG', 'in_reaction_with_GrxC',
    # Drug → Biological Process
    'inhibits_CinBP', 'activates_CaBP', 'affects_CafBP',
    # Gene → Disease
    'marker_or_mechanism_GmD', 'associated_with_GawD', 'treats_GtD',
    # Gene → Biological Process
    'positively_regulates_GprBP', 'negatively_regulates_GnrBP', 'regulates_GrBP',
    # Biological Process → Disease
    'associated_with_BPawD',
    # Drug/Gene → Anatomy → Disease
    'part_of_CpoA', 'associated_with_AawD',
    # Drug → Disease (direct mechanistic, not treatment)
    'marker_or_mechanism_CmD',
}

WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

# Dynamically locate the splits directory (works with any dataset name)
_candidates = list(Path('/kaggle/input').rglob('slice_0'))
if not _candidates:
    raise FileNotFoundError("Could not find slice_0 under /kaggle/input — add the wkb-splits-updated dataset")
SPLITS = _candidates[0].parent
print(f'Splits found at : {SPLITS}')
print(f'Splits to run   : {SPLITS_TO_RUN}')
print(f'Bio relations   : {len(BIO_RELATIONS)}')
print(f'Debug N         : {DEBUG_N}')

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from tqdm import tqdm

MAX_TRACES = 200  # cap path trace expansion to prevent combinatorial explosion

class ProbCBR:
    """
    Probabilistic Case-Based Reasoning — Das et al., EMNLP Findings 2020.

    Paper steps implemented here:
      §2.2.1  Entity representation: binary outgoing-relation-type vectors
      §2.2.2  Entity clustering: KMeans on training drugs only
      §2.2.1  Contextual entity retrieval: cosine k-NN within cluster
      §2.2.3  Path prior  P(p | c, r_q)       — Eq. 2
      §2.2.3  Path precision P(e2 | p, c, r_q) — Eq. 3
      §2.1    Inference score = Σ_p P(p|c) × P(e2|p,c)  — Eq. 1

    MIND adaptation: BFS restricted to BIO_RELATIONS for tractability on 9.65M edge graph.
    Output: top-10 predicted diseases + mechanistic paths per drug.
    """

    def __init__(self, k=10, max_path_len=3, n_clusters=10, min_path_freq=1, seed=42):
        self.k               = k
        self.max_path_len    = max_path_len
        self.n_clusters      = n_clusters
        self.min_path_freq   = min_path_freq
        self.seed            = seed
        self.bio_graph       = {}
        self.entity_vectors  = {}
        self.entity_clusters = {}
        self.cluster_centroids = None
        self.relation_to_id  = {}
        self.drug_paths      = {}
        self.cluster_drugs   = {}

    def _build_bio_graph(self, all_triples):
        g = defaultdict(list)
        for h, r, t in all_triples:
            if r in BIO_RELATIONS:
                g[h].append((r, t))
        self.bio_graph = dict(g)
        n_edges = sum(len(v) for v in self.bio_graph.values())
        print(f'    bio_graph: {n_edges:,} edges, {len(self.bio_graph):,} head entities')

    def _build_entity_vectors(self, all_triples, all_entities):
        rels = sorted(set(r for _, r, _ in all_triples))
        self.relation_to_id = {r: i for i, r in enumerate(rels)}
        dim = len(rels)
        outgoing = defaultdict(set)
        for h, r, _ in all_triples:
            outgoing[h].add(r)
        for entity in all_entities:
            vec = np.zeros(dim, dtype=np.float32)
            for r in outgoing.get(entity, set()):
                vec[self.relation_to_id[r]] = 1.0
            self.entity_vectors[entity] = vec
        print(f'    Entity vectors: {len(self.entity_vectors):,} entities, dim={dim}')

    def _cluster_entities(self, training_drugs):
        ents  = list(training_drugs)
        X     = np.array([self.entity_vectors.get(e, np.zeros(len(self.relation_to_id)))
                          for e in ents], dtype=np.float32)
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1
        X_norm = X / norms
        n_clusters = min(self.n_clusters, len(ents))
        km = MiniBatchKMeans(n_clusters=n_clusters, random_state=self.seed, n_init=5)
        labels = km.fit_predict(X_norm)
        self.entity_clusters   = {e: int(l) for e, l in zip(ents, labels)}
        self.cluster_centroids = km.cluster_centers_
        sizes = defaultdict(int)
        for l in labels:
            sizes[l] += 1
        print(f'    Clusters: {n_clusters}, training drugs per cluster: '
              f'min={min(sizes.values())}, max={max(sizes.values())}, avg={sum(sizes.values())/len(sizes):.0f}')

    def _assign_cluster(self, entity):
        if entity in self.entity_clusters:
            return self.entity_clusters[entity]
        vec  = self.entity_vectors.get(entity, np.zeros(len(self.relation_to_id)))
        norm = np.linalg.norm(vec)
        if norm == 0 or self.cluster_centroids is None:
            return 0
        sims = np.dot(self.cluster_centroids, vec / norm)
        return int(np.argmax(sims))

    def _get_paths(self, entity):
        paths = defaultdict(set)
        queue = [(entity, ())]
        seen  = {(entity, ())}
        while queue:
            curr, path = queue.pop(0)
            if len(path) >= self.max_path_len:
                continue
            for rel, nb in self.bio_graph.get(curr, []):
                new_path = path + (rel,)
                paths[new_path].add(nb)
                if (nb, new_path) not in seen:
                    seen.add((nb, new_path))
                    queue.append((nb, new_path))
        return {k: list(v) for k, v in paths.items()}

    def _precompute_drug_paths(self, ind_triples, query_relation):
        ind_by_drug = defaultdict(set)
        for h, r, t in ind_triples:
            if r == query_relation:
                ind_by_drug[h].add(t)
        for drug, correct_diseases in tqdm(ind_by_drug.items(), desc='    Mining paths'):
            paths = self._get_paths(drug)
            self.drug_paths[drug] = {'paths': paths, 'correct': correct_diseases}
            cluster = self.entity_clusters.get(drug, 0)
            self.cluster_drugs.setdefault(cluster, []).append(drug)
        n_total = sum(len(info['paths']) for info in self.drug_paths.values())
        print(f'    {n_total:,} path types across {len(self.drug_paths):,} training drugs')

    def _find_contextual_entities(self, drug):
        cluster    = self._assign_cluster(drug)
        candidates = self.cluster_drugs.get(cluster, [])
        if not candidates:
            return []
        q_vec  = self.entity_vectors.get(drug, np.zeros(len(self.relation_to_id)))
        q_norm = np.linalg.norm(q_vec)
        if q_norm == 0:
            return candidates[:self.k]
        sims = []
        for ctx in candidates:
            d_vec  = self.entity_vectors.get(ctx, np.zeros(len(self.relation_to_id)))
            d_norm = np.linalg.norm(d_vec)
            sim    = float(np.dot(q_vec, d_vec) / (q_norm * d_norm)) if d_norm > 0 else 0.0
            sims.append((sim, ctx))
        sims.sort(reverse=True)
        return [d for _, d in sims[:self.k]]

    def predict_one(self, drug):
        contextual = self._find_contextual_entities(drug)
        if not contextual:
            return {}, {}

        path_freq_all     = defaultdict(int)
        path_freq_correct = defaultdict(int)
        for ctx in contextual:
            info = self.drug_paths[ctx]
            for path_type, ends in info['paths'].items():
                path_freq_all[path_type]     += len(ends)
                path_freq_correct[path_type] += sum(1 for e in ends if e in info['correct'])

        total_correct = sum(path_freq_correct.values()) or 1

        scores    = defaultdict(float)
        best_path = {}

        for path_type in path_freq_all:
            freq_all     = path_freq_all[path_type]
            freq_correct = path_freq_correct[path_type]
            if freq_all < self.min_path_freq or freq_correct == 0:
                continue

            precision    = freq_correct / freq_all
            prior        = freq_correct / total_correct
            contribution = prior * precision

            # Walk path from query drug, capping traces to avoid combinatorial explosion
            curr_traces = [(drug, [drug])]
            valid = True
            for rel in path_type:
                nxt_traces = []
                for node, trace in curr_traces:
                    for r, nb in self.bio_graph.get(node, []):
                        if r == rel:
                            nxt_traces.append((nb, trace + [nb]))
                if not nxt_traces:
                    valid = False
                    break
                curr_traces = nxt_traces[:MAX_TRACES]  # cap to prevent explosion

            if valid:
                for candidate, trace in curr_traces:
                    scores[candidate] += contribution
                    path_str = ' '.join(
                        f'{trace[i]} --[{path_type[i]}]-->' for i in range(len(path_type))
                    ) + f' {trace[-1]}'
                    if candidate not in best_path or contribution > best_path[candidate][0]:
                        best_path[candidate] = (contribution, path_str)

        return dict(scores), {c: v[1] for c, v in best_path.items()}

    def predict(self, test_queries, n_ents, top_k=10):
        rows = []
        for drug, rel, exp_dis in tqdm(test_queries, desc='  Predicting'):
            scores, path_strings = self.predict_one(drug)
            row = {'drug': drug, 'expected_disease': exp_dis}
            if not scores:
                row['rank']            = n_ents
                row['reciprocal_rank'] = 1.0 / n_ents
                for k in range(1, top_k + 1):
                    row[f'top{k}_disease'] = ''
                    row[f'top{k}_path']    = ''
            else:
                sd   = sorted(scores, key=scores.get, reverse=True)
                rank = sd.index(exp_dis) + 1 if exp_dis in sd else n_ents
                row['rank']            = rank
                row['reciprocal_rank'] = 1.0 / rank
                for k in range(1, top_k + 1):
                    if k - 1 < len(sd):
                        disease = sd[k - 1]
                        row[f'top{k}_disease'] = disease
                        row[f'top{k}_path']    = path_strings.get(disease, '')
                    else:
                        row[f'top{k}_disease'] = ''
                        row[f'top{k}_path']    = ''
            rows.append(row)
        return pd.DataFrame(rows)

    def fit(self, all_triples, ind_triples, all_entities, query_relation):
        print('  [1/4] Building bio_graph...')
        self._build_bio_graph(all_triples)
        print('  [2/4] Building entity vectors...')
        self._build_entity_vectors(all_triples, all_entities)
        print('  [3/4] Clustering training drugs...')
        training_drugs = {h for h, r, _ in ind_triples if r == query_relation}
        self._cluster_entities(training_drugs)
        print('  [4/4] Mining paths for training drugs...')
        self._precompute_drug_paths(ind_triples, query_relation)
        return self

In [ ]:
# ── Cell 3: Load data + Fit (bio_graph, entity vectors, clustering, path mining) ──
# Run this once. Model is saved to disk so Cell 4 can reload without re-fitting.

import pickle, time

PRED_DIR  = WORK / 'predictions' / 'ProbCBR'
MODEL_DIR = WORK / 'models' / 'ProbCBR'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

models = {}   # slice_i -> (fitted ProbCBR model, n_entities)

total_start = time.time()

for i in SPLITS_TO_RUN:
    sl       = SPLITS / f'slice_{i}'
    pkl_path = MODEL_DIR / f'slice_{i}.pkl'
    t0       = time.time()

    # Load from disk if already fitted
    if pkl_path.exists():
        print(f'\nLoading ProbCBR — slice_{i} from {pkl_path}')
        with open(pkl_path, 'rb') as f:
            models[i] = pickle.load(f)
        print(f'  Loaded in {time.time()-t0:.1f}s')
        continue

    print(f'\n{"="*50}\nFitting ProbCBR — slice_{i}\n{"="*50}')

    print('  Loading data...')
    all_triples  = [(r.h, r.r, r.t) for r in
        pd.read_csv(sl/'kge_train.tsv', sep='\t', header=None, names=['h','r','t']).itertuples()]
    ind_triples  = [(r.h, r.r, r.t) for r in
        pd.read_csv(sl/'ind_train.tsv', sep='\t', header=None, names=['h','r','t']).itertuples()]
    all_entities = pd.read_csv(sl/'entities.txt', header=None)[0].tolist()
    print(f'  KG triples: {len(all_triples):,} | Ind triples: {len(ind_triples):,} | Entities: {len(all_entities):,}')

    model = ProbCBR(k=10, max_path_len=3, n_clusters=10, min_path_freq=1)
    model.fit(all_triples, ind_triples, all_entities, INDICATION_REL)
    models[i] = (model, len(all_entities))

    # Diagnostics
    print('\n  === Diagnostics ===')
    print(f'  Clusters with training drugs : {len(model.cluster_drugs)} / {model.n_clusters}')
    sizes = [len(v) for v in model.cluster_drugs.values()]
    print(f'  Training drugs per cluster   : min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)/len(sizes):.0f}')
    with_paths = sum(1 for d in model.drug_paths if model.drug_paths[d]['paths'])
    print(f'  Training drugs with paths    : {with_paths} / {len(model.drug_paths)}')

    # Save to disk
    with open(pkl_path, 'wb') as f:
        pickle.dump(models[i], f)
    print(f'  Saved model → {pkl_path}')

    elapsed = time.time() - t0
    print(f'  Fit done in {elapsed/60:.1f} min')

print(f'\nAll fits done in {(time.time()-total_start)/60:.1f} min')
print('Now run Cell 4 to generate predictions.')

In [ ]:
# ── Cell 4: Predict (re-run freely without re-fitting) ──
# DEBUG_N = 50   → quick debug, no files saved
# DEBUG_N = None → full run, saves predictions + path_lookup TSVs

import pickle, time
from collections import defaultdict
from tqdm import tqdm

TOP_K     = 50   # top predicted diseases saved as columns in predictions TSV
TOP_PATHS = 10   # top paths saved per (drug, disease) pair in path_lookup TSV

# ── Load model from disk if not in memory ────────────────────────────────────
MODEL_DIR = WORK / 'models' / 'ProbCBR'
for i in SPLITS_TO_RUN:
    if i not in models:
        pkl_path = MODEL_DIR / f'slice_{i}.pkl'
        if pkl_path.exists():
            print(f'Loading slice_{i} from {pkl_path}...')
            with open(pkl_path, 'rb') as f:
                models[i] = pickle.load(f)
            print(f'  Loaded.')
        else:
            print(f'slice_{i} not in memory and no saved model — run Cell 3 first')

# ── Metrics helper ───────────────────────────────────────────────────────────

def print_metrics(df, split_name):
    mrr    = df.reciprocal_rank.mean()
    hits1  = (df['rank'] <= 1).mean()
    hits3  = (df['rank'] <= 3).mean()
    hits5  = (df['rank'] <= 5).mean()
    hits10 = (df['rank'] <= 10).mean()
    print(f'  {split_name} MRR={mrr:.4f}  Hits@1={hits1:.4f}  Hits@3={hits3:.4f}  Hits@5={hits5:.4f}  Hits@10={hits10:.4f}')

# ── Fast lookup structures ───────────────────────────────────────────────────

def build_rel_index(bio_graph):
    idx = defaultdict(lambda: defaultdict(list))
    for node, edges in bio_graph.items():
        for rel, nb in edges:
            idx[rel][node].append(nb)
    return idx

def build_cluster_matrices(model):
    dim = len(model.relation_to_id)
    matrices = {}
    for cid, drugs in model.cluster_drugs.items():
        vecs = np.array([model.entity_vectors.get(d, np.zeros(dim, dtype=np.float32))
                         for d in drugs], dtype=np.float32)
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        matrices[cid] = (drugs, vecs / norms)
    return matrices, dim

def fast_contextual(drug, model, cluster_matrices, dim):
    cluster = model._assign_cluster(drug)
    if cluster not in cluster_matrices:
        return []
    drugs_list, normed = cluster_matrices[cluster]
    q_vec = model.entity_vectors.get(drug, np.zeros(dim, dtype=np.float32))
    q_norm = np.linalg.norm(q_vec)
    if q_norm == 0:
        return drugs_list[:model.k]
    sims = normed @ (q_vec / q_norm)
    top_idx = np.argsort(sims)[::-1][:model.k]
    return [drugs_list[j] for j in top_idx]

def find_path_str(drug, path_type, candidate, rel_index):
    curr = {drug: (drug,)}
    for rel in path_type:
        nxt = {}
        for node, trace in curr.items():
            for nb in rel_index[rel].get(node, []):
                if nb not in nxt:
                    nxt[nb] = trace + (nb,)
        curr = nxt
        if not curr:
            return ''
    trace = curr.get(candidate)
    if not trace:
        return ''
    return ' '.join(
        f'{trace[j]} --[{path_type[j]}]-->' for j in range(len(path_type))
    ) + f' {candidate}'

def fast_predict_one(drug, model, rel_index, cluster_matrices, dim):
    contextual = fast_contextual(drug, model, cluster_matrices, dim)
    if not contextual:
        return {}, {}, contextual

    path_freq_all     = defaultdict(int)
    path_freq_correct = defaultdict(int)
    for ctx in contextual:
        info = model.drug_paths[ctx]
        for path_type, ends in info['paths'].items():
            path_freq_all[path_type]     += len(ends)
            path_freq_correct[path_type] += sum(1 for e in ends if e in info['correct'])

    total_correct = sum(path_freq_correct.values()) or 1

    scores         = defaultdict(float)
    all_path_types = defaultdict(list)

    for path_type in path_freq_all:
        freq_all     = path_freq_all[path_type]
        freq_correct = path_freq_correct[path_type]
        if freq_all < model.min_path_freq or freq_correct == 0:
            continue

        precision    = freq_correct / freq_all
        prior        = freq_correct / total_correct
        contribution = prior * precision

        curr_nodes = {drug}
        valid = True
        for rel in path_type:
            nxt_nodes = set()
            for node in curr_nodes:
                nxt_nodes.update(rel_index[rel].get(node, []))
            if not nxt_nodes:
                valid = False
                break
            if len(nxt_nodes) > MAX_TRACES:
                nxt_nodes = set(list(nxt_nodes)[:MAX_TRACES])
            curr_nodes = nxt_nodes

        if valid:
            for candidate in curr_nodes:
                scores[candidate] += contribution
                all_path_types[candidate].append((contribution, path_type))

    return dict(scores), dict(all_path_types), contextual


PRED_DIR = WORK / 'predictions' / 'ProbCBR'

for i in SPLITS_TO_RUN:
    if i not in models:
        print(f'slice_{i} not available — run Cell 3 first'); continue

    model, n_ents = models[i]
    sl     = SPLITS / f'slice_{i}'
    outdir = PRED_DIR / f'slice_{i}'
    outdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f'\n{"="*50}\nPredicting — slice_{i}\n{"="*50}')

    rel_index             = build_rel_index(model.bio_graph)
    cluster_matrices, dim = build_cluster_matrices(model)
    print(f'  rel_index: {sum(len(v) for v in rel_index.values()):,} entries | '
          f'cluster matrices: {len(cluster_matrices)} clusters')

    all_path_rows = []

    for split in ['test', 'valid']:
        ev = [(r.h, r.r, r.t) for r in
              pd.read_csv(sl/f'ind_{split}.tsv', sep='\t', header=None, names=['h','r','t']).itertuples()]

        if DEBUG_N:
            ev = ev[:DEBUG_N]
            print(f'\n  [{split}] Debug mode — first {DEBUG_N} queries (not saving)')
        else:
            done = outdir / f'predictions_{split}.tsv'
            if done.exists():
                print(f'  SKIP {split} (already exists)')
                continue
            print(f'\n  [{split}] Running {len(ev)} queries...')

        n_no_ctx, n_no_score = 0, 0
        rows = []
        for drug, rel, exp_dis in tqdm(ev, desc=f'  {split}'):
            scores, all_path_types, ctx = fast_predict_one(drug, model, rel_index, cluster_matrices, dim)
            if not ctx: n_no_ctx += 1

            row = {'drug': drug, 'expected_disease': exp_dis}
            if not scores:
                n_no_score += 1
                row['rank']            = n_ents
                row['reciprocal_rank'] = 1.0 / n_ents
                for k in range(1, TOP_K + 1):
                    row[f'top{k}_disease'] = ''
            else:
                sd   = sorted(scores, key=scores.get, reverse=True)
                rank = sd.index(exp_dis) + 1 if exp_dis in sd else n_ents
                row['rank']            = rank
                row['reciprocal_rank'] = 1.0 / rank
                for k in range(1, TOP_K + 1):
                    row[f'top{k}_disease'] = sd[k-1] if k-1 < len(sd) else ''

                candidates_for_paths = sd[:TOP_K]
                if exp_dis not in candidates_for_paths:
                    candidates_for_paths = candidates_for_paths + [exp_dis]

                for disease in candidates_for_paths:
                    if disease not in all_path_types:
                        continue
                    ranked_paths = sorted(all_path_types[disease], reverse=True)[:TOP_PATHS]
                    for path_rank, (path_score, path_type) in enumerate(ranked_paths, 1):
                        pstr = find_path_str(drug, path_type, disease, rel_index)
                        if pstr:
                            all_path_rows.append({
                                'drug':        drug,
                                'disease':     disease,
                                'path_rank':   path_rank,
                                'path_score':  round(path_score, 6),
                                'path':        pstr,
                                'is_expected': disease == exp_dis,
                            })

            rows.append(row)

        df_out = pd.DataFrame(rows)
        print_metrics(df_out, split)
        print(f'  No contextual entities : {n_no_ctx}/{len(ev)}')
        print(f'  No scores              : {n_no_score}/{len(ev)}')

        if not DEBUG_N:
            df_out.to_csv(outdir / f'predictions_{split}.tsv', sep='\t', index=False)
            print(f'  Saved → {outdir}/predictions_{split}.tsv')

    if not DEBUG_N and all_path_rows:
        path_lookup = pd.DataFrame(all_path_rows)
        path_lookup.to_csv(outdir / 'path_lookup.tsv', sep='\t', index=False)
        print(f'\n  Path lookup: {len(path_lookup):,} rows '
              f'({path_lookup.groupby(["drug","disease"]).ngroups:,} drug-disease pairs)')

    print(f'  Done in {(time.time()-t0)/60:.1f} min')